# 05 · Inference — Score Upcoming Matches

This notebook demonstrates the **full production workflow** as it runs in Streamlit:

1. **Download the most up-to-date historical data** from football-data.co.uk (same seasons used online)
2. **Feature-engineer** the historical data (rolling windows need match history as context)
3. **Scrape upcoming fixtures** from TalkSport
4. **Align team names** — dictionary first, fuzzy matching as fallback for any new names
5. **Append upcoming matches** to the historical dataset and run feature engineering
6. **Predict** with the trained Random Forest model

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
from fuzzywuzzy import process

from src.data.extract import (
    load_upcoming_fixtures,
    FIXTURE_PAGES,
    TEAM_NAME_FIXES,
)
from src.features.engineering import (
    assign_season,
    select_raw_columns,
    build_features,
    FEATURE_COLUMNS,
    INV_LABEL_MAP,
)
from src.models.predict import load_model, predict

MODEL_PATH = pathlib.Path("../models/rf_match_predict_model.pkl")

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


---
## Step 1 · Download most up-to-date historical data

In production (Streamlit) we only load the **two most recent seasons** per
league — that gives enough history for the rolling windows (up to 38 matches)
while keeping download time short.

The URLs always point to the **live-updated CSVs** on football-data.co.uk,
so every time you run this you get results through the latest matchday.

In [ ]:
DATA_SOURCES = {
    "LaLiga 23/24": "https://www.football-data.co.uk/mmz4281/2324/SP1.csv",
    "LaLiga 24/25": "https://www.football-data.co.uk/mmz4281/2425/SP1.csv",
    "LaLiga 25/26": "https://www.football-data.co.uk/mmz4281/2526/SP1.csv",
    "EPL 23/24":    "https://www.football-data.co.uk/mmz4281/2324/E0.csv",
    "EPL 24/25":    "https://www.football-data.co.uk/mmz4281/2425/E0.csv",
    "EPL 25/26":    "https://www.football-data.co.uk/mmz4281/2526/E0.csv",
}

frames = []
for label, url in DATA_SOURCES.items():
    df = pd.read_csv(url)
    print(f"  {label}: {len(df)} matches")
    frames.append(df)

history = pd.concat(frames, ignore_index=True)
history["Date"] = pd.to_datetime(history["Date"], dayfirst=True, errors="coerce")
history["Season"] = history["Date"].apply(assign_season)
history = select_raw_columns(history)

print(f"\nTotal historical matches loaded: {len(history):,}")
print(f"Date range: {history['Date'].min().date()} → {history['Date'].max().date()}")
history.tail(3)

  LaLiga 23/24: 380 matches
  LaLiga 24/25: 380 matches
  LaLiga 25/26: 290 matches
  EPL 23/24: 380 matches
  EPL 24/25: 380 matches
  EPL 25/26: 309 matches

Total historical matches loaded: 2,119
Date range: 2023-08-11 → 2026-03-22


/var/folders/5d/1xw4nstd6ls9kr2sdhvp91l40000gn/T/ipykernel_43091/3392421146.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  history["Season"] = history["Date"].apply(assign_season)


,Div,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
2116,E0,2025/2026,2026-03-22,Newcastle,Sunderland,1,2,A,1,0,...,4,7,8,19,9,5,1,6,0,0
2117,E0,2025/2026,2026-03-22,Aston Villa,West Ham,2,0,H,1,0,...,7,1,3,7,6,6,0,1,0,0
2118,E0,2025/2026,2026-03-22,Tottenham,Nott'm Forest,0,3,A,0,1,...,3,7,13,13,13,3,1,2,0,0


---
## Step 2 · Scrape upcoming fixtures

Scrapes the TalkSport fixture pages for La Liga and the Premier League.
Returns a minimal DataFrame with Div, Date, HomeTeam, AwayTeam — match
stats are NaN since these games haven't been played yet.

In [ ]:
upcoming_raw = load_upcoming_fixtures()
print(f"Scraped {len(upcoming_raw)} upcoming fixtures\n")
upcoming_raw

Scraped 20 upcoming fixtures



,Div,Date,HomeTeam,AwayTeam
0,SP1,2026-03-29,Rayo,Elche
1,SP1,2026-03-29,Real Sociedad,Levante
2,SP1,2026-03-29,Mallorca,Real Madrid
3,SP1,2026-03-29,Betis,Espanyol
4,SP1,2026-03-29,Atlético,Barcelona
5,SP1,2026-03-29,Getafe,Athletic
6,SP1,2026-03-29,Valencia,Celta
7,SP1,2026-03-29,Oviedo,Sevilla
8,SP1,2026-03-29,Alavés,Osasuna
9,SP1,2026-03-29,Girona,Villarreal


---
## Step 3 · Align team names

Scraped names don't always match the historical dataset.
We fix them in two passes:

1. **Dictionary lookup** — known mismatches (fast, deterministic)
2. **Fuzzy matching** — catches any new names that aren't in the dictionary

In [ ]:
# Build the reference list of canonical team names from the historical data
valid_names = sorted(history["HomeTeam"].dropna().unique().tolist())
print(f"{len(valid_names)} canonical team names in historical data\n")

# --- Pass 1: Dictionary ---
# These are known name differences between TalkSport and football-data.co.uk
print("Known name fixes (dictionary):")
for scraped, canonical in TEAM_NAME_FIXES.items():
    print(f"  {scraped:25s} → {canonical}")

upcoming = upcoming_raw.copy()
upcoming["HomeTeam"] = upcoming["HomeTeam"].replace(TEAM_NAME_FIXES)
upcoming["AwayTeam"] = upcoming["AwayTeam"].replace(TEAM_NAME_FIXES)

51 canonical team names in historical data

Known name fixes (dictionary):
  Athletic Club             → Ath Bilbao
  Athletic                  → Ath Bilbao
  Atlético                  → Atl Madrid
  Atletico Madrid           → Atl Madrid
  Atletico                  → Atl Madrid
  Rayo                      → Vallecano
  Rayo Vallecano            → Vallecano
  Espanyol                  → Espanol
  Real Sociedad             → Sociedad
  Alavés                    → Alaves
  Spurs                     → Tottenham
  Man Utd                   → Man United
  Nottm Forest              → Nott'm Forest


In [ ]:
# --- Pass 2: Fuzzy matching for any remaining mismatches ---
valid_set = set(valid_names)

def fuzzy_fix(name: str) -> str:
    if name in valid_set:
        return name
    match, score = process.extractOne(name, valid_names)
    print(f"  Fuzzy matched: {name:25s} → {match} (score {score})")
    return match

print("Fuzzy matching remaining names:")
upcoming["HomeTeam"] = upcoming["HomeTeam"].apply(fuzzy_fix)
upcoming["AwayTeam"] = upcoming["AwayTeam"].apply(fuzzy_fix)
print("  (no output = all names already matched)\n")

upcoming

Fuzzy matching remaining names:
  Fuzzy matched: Atl Madrid                → Ath Madrid (score 90)
  (no output = all names already matched)



,Div,Date,HomeTeam,AwayTeam
0,SP1,2026-03-29,Vallecano,Elche
1,SP1,2026-03-29,Sociedad,Levante
2,SP1,2026-03-29,Mallorca,Real Madrid
3,SP1,2026-03-29,Betis,Espanol
4,SP1,2026-03-29,Ath Madrid,Barcelona
5,SP1,2026-03-29,Getafe,Ath Bilbao
6,SP1,2026-03-29,Valencia,Celta
7,SP1,2026-03-29,Oviedo,Sevilla
8,SP1,2026-03-29,Alaves,Osasuna
9,SP1,2026-03-29,Girona,Villarreal


---
## Step 4 · Feature engineering on combined dataset

We **append the upcoming (unplayed) matches to the historical data** before
running feature engineering. This way, the rolling windows for the upcoming
matches are computed from the team's actual recent form.

Since upcoming matches have no result (FTR is NaN), the rolling stats
naturally exclude them — no data leakage.

In [ ]:
# Save keys to filter upcoming rows back out after feature engineering
upcoming_keys = upcoming[["Div", "Date", "HomeTeam", "AwayTeam"]].copy()

# Combine and run full feature engineering
combined = pd.concat([history, upcoming], ignore_index=True)
combined = build_features(combined, drop_na=False)

# Recover only the upcoming rows
upcoming_engineered = combined.merge(
    upcoming_keys.assign(IsUpcoming=True),
    on=["Div", "Date", "HomeTeam", "AwayTeam"],
    how="inner",
)

print(f"{len(upcoming_engineered)} upcoming fixtures with features")
print(f"\nSample features for first match:")
upcoming_engineered[FEATURE_COLUMNS].iloc[0].to_frame("value").T

20 upcoming fixtures with features

Sample features for first match:


,RollingPointsDif10,HomeTeamPointsLast2Encounters,HomeRollingPoints38,AwayRollingPoints38,RollingPointsDiff38,HomeRollingShotsTarget20,AwayRollingShotsTarget20,HomeRollingShots20,AwayRollingShots20,HomeRollingYellowCards20,...,AwayRollingReceivedGoals20,RollingShotsDif20,RollingShotsTargetDif20,RollingYellowCardsDif20,RollingRedCardsDif20,RollingFoulsDif20,RollingCornersDif20,HomeRank,AwayRank,RankDif
value,-2.0,1.0,76.0,66.0,10.0,96.0,88.0,308.0,283.0,41.0,...,18.0,25.0,8.0,3.0,3.0,33.0,41.0,6.0,2.0,4.0


In [ ]:
# Verify no critical features are missing
null_counts = upcoming_engineered[FEATURE_COLUMNS].isnull().sum()
if null_counts.any():
    print("⚠️  Features with nulls (will be filled with 0 for prediction):")
    print(null_counts[null_counts > 0])
else:
    print("✓ All features populated — no nulls")

✓ All features populated — no nulls


---
## Step 5 · Predict with Random Forest

Load the trained Random Forest model and generate:
- **Predicted result** (H / D / A)
- **Probabilities** for each outcome

In [ ]:
model = load_model(MODEL_PATH)
print(f"✓ Loaded model from {MODEL_PATH}")
print(f"  Type: {type(model).__name__}")

✓ Loaded model from ../models/rf_match_predict_model.pkl
  Type: RandomForestClassifier


In [ ]:
preds = predict(model, upcoming_engineered)

output = pd.concat(
    [upcoming_engineered[["Div", "Date", "HomeTeam", "AwayTeam"]].reset_index(drop=True), preds.reset_index(drop=True)],
    axis=1,
).sort_values(["Div", "Date"]).reset_index(drop=True)

output

,Div,Date,HomeTeam,AwayTeam,PredictedResult,Prob_H,Prob_D,Prob_A
0,E0,2026-03-29,Chelsea,Man City,A,0.315726,0.258627,0.425647
1,E0,2026-03-29,Brentford,Everton,H,0.393106,0.294458,0.312436
2,E0,2026-03-29,Arsenal,Bournemouth,H,0.547591,0.228745,0.223664
3,E0,2026-03-29,West Ham,Wolves,H,0.382065,0.268290,0.349645
4,E0,2026-03-29,Sunderland,Tottenham,H,0.377915,0.269289,0.352797
5,E0,2026-03-29,Burnley,Brighton,A,0.232820,0.219004,0.548175
6,E0,2026-03-29,Liverpool,Fulham,H,0.583061,0.240478,0.176461
7,E0,2026-03-29,Crystal Palace,Newcastle,A,0.297088,0.300960,0.401952
8,E0,2026-03-29,Nott'm Forest,Aston Villa,A,0.319160,0.263886,0.416954
9,E0,2026-03-29,Man United,Leeds,H,0.697819,0.181167,0.121014


---
## Results by League

Formatted output, matching what the Streamlit dashboard shows.

In [ ]:
for div, label in [("SP1", "La Liga"), ("E0", "Premier League")]:
    subset = output[output["Div"] == div]
    if subset.empty:
        continue

    print(f"\n{'=' * 70}")
    print(f"  {label}")
    print(f"{'=' * 70}")

    for _, row in subset.iterrows():
        home = row["HomeTeam"]
        away = row["AwayTeam"]
        pred = row["PredictedResult"]
        ph = row["Prob_H"]
        pd_ = row["Prob_D"]
        pa = row["Prob_A"]
        
        winner = home if pred == "H" else (away if pred == "A" else "Draw")
        print(f"  {home:20s} vs {away:20s}  →  {pred}  ({ph:.0%} / {pd_:.0%} / {pa:.0%})  [{winner}]")

    print()


  La Liga
  Alaves               vs Osasuna               →  H  (41% / 33% / 26%)  [Alaves]
  Mallorca             vs Real Madrid           →  A  (11% / 17% / 72%)  [Real Madrid]
  Betis                vs Espanol               →  H  (48% / 27% / 24%)  [Betis]
  Ath Madrid           vs Barcelona             →  A  (26% / 31% / 43%)  [Barcelona]
  Oviedo               vs Sevilla               →  A  (37% / 24% / 39%)  [Sevilla]
  Valencia             vs Celta                 →  A  (34% / 29% / 38%)  [Celta]
  Getafe               vs Ath Bilbao            →  A  (30% / 27% / 44%)  [Ath Bilbao]
  Vallecano            vs Elche                 →  H  (60% / 26% / 14%)  [Vallecano]
  Girona               vs Villarreal            →  A  (29% / 27% / 44%)  [Villarreal]
  Sociedad             vs Levante               →  H  (55% / 27% / 18%)  [Sociedad]


  Premier League
  Chelsea              vs Man City              →  A  (32% / 26% / 43%)  [Man City]
  Brentford            vs Everton             

---
## Summary

This notebook replicates the **exact same pipeline** that runs inside
`streamlit_app.py`:

| Step | What happens | Where in Streamlit |
|------|-------------|-------------------|
| 1. Download CSVs | `load_match_history()` | Cached with `@st.cache_data` |
| 2. Scrape fixtures | `load_upcoming_fixtures()` | Cached with `@st.cache_data` |
| 3. Name alignment | `align_team_names()` | Dictionary + fuzzy fallback |
| 4. Feature engineering | `build_features()` | Runs on combined history + upcoming |
| 5. Predict | `predict()` | Model loaded with `@st.cache_resource` |

To run the Streamlit app:
```bash
streamlit run streamlit_app.py
```